In [1]:
import torch
import torch.nn as nn

In [2]:
# 코드 4-5 숏컷 연결을 설명하기 위한 신경망

class ExampleNeuralNetwork(nn.Module):
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]), nn.GELU()),
            nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]), nn.GELU()),
            nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]), nn.GELU()),
            nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]), nn.GELU()),
            nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]), nn.GELU()),
        ])

    def forward(self, x):
        for layer in self.layers:
            layer_output = layer(x)
            if self.use_shortcut and x.shape == layer_output.shape:
                x = x + layer_output
            else:
                x = layer_output

        return x

In [3]:
# 숏컷 연결이 없는 신경망을 초기화
# 각 층은 3개의 입력 값을 가진 샘플을 받아 3개의 출력 값을 반환
# 마지막 층은 하나의 값을 반환

layer_sizes = [3, 3, 3, 3, 3, 1]

sample_input =  torch.tensor([
    [1., 0., -1.]
])

torch.manual_seed(123)

model_without_shortcut = ExampleNeuralNetwork(layer_sizes, use_shortcut=False)

# 모델의 역전파에서 그레이디언트를 계산하는 함수를 구현
def print_gradients(model, x):
    output = model(x)
    target = torch.tensor([[0.]])

    loss = nn.MSELoss()
    loss = loss(output, target)
    loss.backward()

    for name, param in model.named_parameters():
        if 'weight' in name:
            print(f"{name}의 평균 그레이디언트는 {param.grad.abs().mean().item()}입니다.")

In [ ]:
# print_gradients 함수를 숏컷 연결이 없는 모델에 적용
print_gradients(model_without_shortcut, sample_input)

# layers.0.0.weight의 평균 그레이디언트는 0.0002017411752603948입니다.
# layers.1.0.weight의 평균 그레이디언트는 0.00012011769285891205입니다.
# layers.2.0.weight의 평균 그레이디언트는 0.0007152435719035566입니다.
# layers.3.0.weight의 평균 그레이디언트는 0.0013988512801006436입니다.
# layers.4.0.weight의 평균 그레이디언트는 0.005049605388194323입니다.

layers.0.0.weight의 평균 그레이디언트는 0.0002017411752603948입니다.
layers.1.0.weight의 평균 그레이디언트는 0.00012011769285891205입니다.
layers.2.0.weight의 평균 그레이디언트는 0.0007152435719035566입니다.
layers.3.0.weight의 평균 그레이디언트는 0.0013988512801006436입니다.
layers.4.0.weight의 평균 그레이디언트는 0.005049605388194323입니다.


In [ ]:
# 숏컷 연결을 가진 모델을 만들어서 비교해 보자
torch.manual_seed(123)

model_with_shortcut = ExampleNeuralNetwork(layer_sizes, use_shortcut=True)

print_gradients(model_with_shortcut, sample_input)

# layers.0.0.weight의 평균 그레이디언트는 0.22186797857284546입니다.
# layers.1.0.weight의 평균 그레이디언트는 0.207092747092247입니다.
# layers.2.0.weight의 평균 그레이디언트는 0.3292388319969177입니다.
# layers.3.0.weight의 평균 그레이디언트는 0.2667772173881531입니다.
# layers.4.0.weight의 평균 그레이디언트는 1.3268064260482788입니다.

layers.0.0.weight의 평균 그레이디언트는 0.22186797857284546입니다.
layers.1.0.weight의 평균 그레이디언트는 0.207092747092247입니다.
layers.2.0.weight의 평균 그레이디언트는 0.3292388319969177입니다.
layers.3.0.weight의 평균 그레이디언트는 0.2667772173881531입니다.
layers.4.0.weight의 평균 그레이디언트는 1.3268064260482788입니다.
